# 02 — Search Engine: FAISS Index, Semantic Search, Summarization & Keywords
### AI Research Paper Intelligence System

This notebook builds the second half of the pipeline, using the cleaned
data and embeddings produced in `01_EDA_and_Embeddings.ipynb`:

1. Load cached embeddings and build a FAISS vector index
2. Run semantic search queries
3. Wrap search into a reusable function
4. Summarize results with a BART-based model
5. Extract keywords with KeyBERT

## 1. Load cached data & embeddings

Loading pre-computed embeddings from disk takes about a second, versus the
~20-30 minutes needed to regenerate them — this is why we cached them in
the previous notebook.

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv")

print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")

print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(embeddings.shape, embeddings.dtype)

## 2. Build the FAISS index

**Why FAISS?** For a corpus this size, comparing a query against every
paper with `cosine_similarity` in a Python loop would be slow. FAISS is a
highly-optimized C++ library (from Meta AI) built specifically for
fast nearest-neighbor search over large vector collections.

**Why normalize embeddings first?** In semantic search, meaning is encoded
in a vector's *direction*, not its length. By scaling every vector to unit
length (L2 normalization), FAISS's fast Inner Product search becomes
mathematically equivalent to Cosine Similarity — giving us the best of
both worlds: correctness *and* speed.

We cache the built index to disk exactly like we cached the embeddings.

In [ ]:
import faiss
import os

index_path = "../data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index...")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index...")
    # Work on a copy -- FAISS normalizes in-place and would otherwise
    # permanently alter our original `embeddings` array.
    faiss_embeddings = embeddings.copy()
    faiss.normalize_L2(faiss_embeddings)

    index = faiss.IndexFlatIP(384)   # Inner Product on normalized vectors = cosine similarity
    index.add(faiss_embeddings)

    os.makedirs("../data", exist_ok=True)
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully!")

print("Papers indexed:", index.ntotal)

## 3. Run a search query

To search, we:
1. Encode the query text with the *same* embedding model used for the papers
2. Normalize the query vector (so it's on equal footing with the indexed vectors)
3. Ask FAISS for the `k` nearest neighbors by inner product (= cosine similarity)

In [ ]:
query = "deep learning for medical image analysis"

query_embedding = model.encode([query])
faiss.normalize_L2(query_embedding)

D, I = index.search(query_embedding, 5)  # D = similarity scores, I = row indices

print("Scores (D):", D)
print("Indices (I):", I)

In [ ]:
# Inspect the top result
print(df.iloc[I[0][0]]['title'])

## 4. Wrap search into a reusable function

`D` and `I` are 2D arrays (shape `(1, k)`) because FAISS supports batched
queries. Since we only pass one query at a time, we index into `[0]` to get
the flat list of scores/indices, then `zip()` them together to loop through
score + row number in lockstep.

In [ ]:
def search_paper(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity Score:", score)
        print("Title:", df.iloc[idx]['title'])
        print("Abstract:", df.iloc[idx]['abstract'][:500])
        print()  # spacer between papers

    return D, I

D, I = search_paper(query)

## 5. Summarize results with BART

Reading full abstracts for every result is slow. We use a distilled BART
model (`distilbart-cnn-12-6` — ~300MB, much lighter than full BART) to
generate a short, readable summary of each paper's abstract.

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=0   # set to -1 to force CPU if no GPU is available
)

In [ ]:
sample_summary = summarizer(df.iloc[10466]['abstract'], max_length=120, min_length=40)
print(sample_summary[0]["summary_text"])

## 6. Combine search + summarization

Now we extend `search_paper` into `search_and_summarize`: for every result,
fetch the title/abstract from the dataframe *and* generate an AI summary.

In [ ]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)

    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()

        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40)
        print("AI SUMMARY:")
        print(summary[0]["summary_text"])
        print("-" * 80)

search_and_summarize("Deep learning in medical science", k=5)

## 7. Extract keywords with KeyBERT

Beyond a summary, it's useful to see the core *topics* of a paper at a
glance. KeyBERT uses the same embedding model to find the phrases in a text
that are most semantically representative of the whole document.

We reuse our existing `model` (the same `all-MiniLM-L6-v2` used for search)
so keywords live in the exact same semantic space as everything else.

In [ ]:
from keybert import KeyBERT

kw_model = KeyBERT(model=model)
type(kw_model)

In [ ]:
text = df.iloc[26063]['abstract']
keywords = kw_model.extract_keywords(text)
print(keywords)

By default, KeyBERT extracts single words, which can split apart
meaningful multi-word terms (e.g. "deep" and "learning" separately instead
of "deep learning"). Setting `keyphrase_ngram_range=(1, 3)` allows phrases
of up to 3 words, capturing more natural, meaningful key phrases.

In [ ]:
final_keywords = kw_model.extract_keywords(
    text,
    keyphrase_ngram_range=(1, 3),
    stop_words="english"
)
final_keywords

---
## Summary

We now have a complete pipeline:

`query → FAISS semantic search → top-k papers → BART summary + KeyBERT keywords`

This exact logic has been refactored into reusable modules in `../src/`:
- `src/search_engine.py` → `PaperSearchEngine` class (search, summarize, extract_keywords, full_report)
- `src/app.py` → Streamlit web app built on top of `PaperSearchEngine`

To launch the interactive app:

```bash
streamlit run ../src/app.py
```